# SAM 2 Segmentation — Run Once in Colab

**What this notebook does:** Loads your local annotations from `hummingbird_keyframes.json` and runs SAM 2 to segment crown, back, and tail across the full video.

**Before running:**
1. Complete annotation locally in `hummingbird_annotate_local.ipynb`
2. Upload `hummingbird_keyframes.json` to your Google Drive `HummingBird Project/` folder

**Runtime:** Runtime > Change runtime type > **T4 GPU**

**After running:** Download `hummingbird_masks.npz` from Drive → run `hummingbird_analysis_local.ipynb` locally.

In [ ]:
# ── Cell 1: Install SAM 2 + checkpoint ───────────────────────────────────────
!pip install -q git+https://github.com/facebookresearch/sam2.git

import os
if not os.path.exists('/content/sam2_hiera_small.pt'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -P /content/
    print('Checkpoint downloaded.')
else:
    print('Checkpoint already present.')

In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from contextlib import nullcontext

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {device}')

PyTorch 2.10.0+cu128  |  device: cuda


In [3]:
# ── Cell 3: Mount Drive and locate video ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [5]:
VIDEO_PATH = '/content/drive/MyDrive/HummingBird Project/Hummingbird.mov'  # <-- adjust if needed
MASKS_OUT  = '/content/drive/MyDrive/HummingBird Project/hummingbird_masks.npz'

assert os.path.exists(VIDEO_PATH), f'Video not found: {VIDEO_PATH}'
print(f'Video found: {VIDEO_PATH}')

Video found: /content/drive/MyDrive/HummingBird Project/Hummingbird.mov


In [6]:
# ── Cell 4: Extract frames ────────────────────────────────────────────────────
FRAMES_DIR = '/content/frames'
os.makedirs(FRAMES_DIR, exist_ok=True)

cap          = cv2.VideoCapture(VIDEO_PATH)
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{width}x{height} @ {fps:.1f} FPS — {total_frames} frames ({total_frames/fps:.1f}s)')

saved = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    cv2.imwrite(f'{FRAMES_DIR}/{saved:05d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    saved += 1
cap.release()
print(f'Extracted {saved} frames.')

446x442 @ 25.0 FPS — 2642 frames (105.8s)
Extracted 2628 frames.


In [ ]:
# ── Cell 5: Load keyframes from Google Drive ──────────────────────────────────
import json

KEYFRAMES_PATH = '/content/drive/MyDrive/HummingBird Project/hummingbird_keyframes.json'
assert os.path.exists(KEYFRAMES_PATH), (
    f'Keyframes not found: {KEYFRAMES_PATH}\n'
    '  → Run hummingbird_annotate_local.ipynb on your Mac first,\n'
    '    then upload hummingbird_keyframes.json to Google Drive.'
)

with open(KEYFRAMES_PATH) as f:
    KEYFRAMES = json.load(f)

print(f'Loaded {len(KEYFRAMES)} keyframes:')
for kf in KEYFRAMES:
    parts_summary = ', '.join(
        f'{p}={kf[p]}' if kf[p] is not None else f'{p}=hidden'
        for p in ['crown', 'back', 'tail']
    )
    print(f'  frame {kf["frame"]}: {parts_summary}')

In [ ]:
# ── Cell 7: Load SAM 2 ────────────────────────────────────────────────────────
from sam2.build_sam import build_sam2_video_predictor

def autocast_ctx(dev):
    return torch.autocast('cuda', dtype=torch.bfloat16) if dev == 'cuda' else nullcontext()

predictor = build_sam2_video_predictor(
    config_file='sam2_hiera_s.yaml',
    ckpt_path='/content/sam2_hiera_small.pt',
    device=device,
)
print(f'SAM 2 loaded on {device}.')

In [ ]:
# ── Cell 8: Add all keyframe prompts and propagate ────────────────────────────
import gc

PART_IDS   = {'crown': 1, 'back': 2, 'tail': 3}
ID_TO_PART = {v: k for k, v in PART_IDS.items()}

cx_mid = width  // 2
cy_mid = height // 2

with torch.inference_mode(), autocast_ctx(device):
    state = predictor.init_state(video_path=FRAMES_DIR)
    predictor.reset_state(state)

    for kf in KEYFRAMES:
        fidx = kf['frame']
        for part, obj_id in PART_IDS.items():
            pt = kf[part]
            if pt is not None:
                predictor.add_new_points_or_box(
                    inference_state=state,
                    frame_idx=fidx,
                    obj_id=obj_id,
                    points=np.array([pt], dtype=np.float32),
                    labels=np.array([1], dtype=np.int32),
                )
                print(f'  frame {fidx}: {part} visible @ {pt}')
            else:
                predictor.add_new_points_or_box(
                    inference_state=state,
                    frame_idx=fidx,
                    obj_id=obj_id,
                    points=np.array([[cx_mid, cy_mid]], dtype=np.float32),
                    labels=np.array([0], dtype=np.int32),
                )
                print(f'  frame {fidx}: {part} hidden → negative hint')

    anchor = KEYFRAMES[0]['frame']
    video_segments = {}

    print(f'\nForward pass ({anchor} → end)...')
    for fidx, obj_ids, logits in predictor.propagate_in_video(state, start_frame_idx=anchor):
        video_segments[fidx] = {
            ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
            for i, oid in enumerate(obj_ids)
        }
        del logits
        if fidx % 20 == 0:
            print(f'  frame {fidx}')
            gc.collect()
            torch.cuda.empty_cache()

    if anchor > 0:
        print(f'\nBackward pass ({anchor} → 0)...')
        for fidx, obj_ids, logits in predictor.propagate_in_video(state, start_frame_idx=anchor, reverse=True):
            video_segments[fidx] = {
                ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
                for i, oid in enumerate(obj_ids)
            }
            del logits
            if fidx % 20 == 0:
                print(f'  frame {fidx}')
                gc.collect()
                torch.cuda.empty_cache()

print(f'\nDone — {len(video_segments)} frames segmented.')

In [ ]:
# ── Cell 8: Propagate (chunked to stay within RAM) ────────────────────────────
import gc, os, shutil
from pathlib import Path

PART_IDS   = {'crown': 1, 'back': 2, 'tail': 3}
ID_TO_PART = {v: k for k, v in PART_IDS.items()}
cx_mid, cy_mid = width // 2, height // 2

CHUNK_SIZE = 300
CHUNK_DIR  = '/content/frames_chunk'

kf_by_frame = {kf['frame']: kf for kf in KEYFRAMES}
all_frames  = sorted(int(f.stem) for f in Path(FRAMES_DIR).glob('*.jpg'))
chunks      = [all_frames[i:i + CHUNK_SIZE] for i in range(0, len(all_frames), CHUNK_SIZE)]

video_segments = {}
carry_masks    = None

print(f'{len(all_frames)} frames  →  {len(chunks)} chunks of ≤{CHUNK_SIZE}\n')

for ci, chunk in enumerate(chunks):
    chunk_start = chunk[0]
    chunk_kfs   = [f for f in chunk if f in kf_by_frame]
    anchor_orig = chunk_kfs[0] if chunk_kfs else chunk[0]
    anchor_rel  = anchor_orig - chunk_start

    label = f'keyframes {chunk_kfs}' if chunk_kfs else 'carry-over'
    print(f'Chunk {ci + 1}/{len(chunks)}: frames {chunk[0]}–{chunk[-1]}  ({label})')

    tmp = Path(CHUNK_DIR)
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    for i, fidx in enumerate(chunk):
        os.symlink(f'{FRAMES_DIR}/{fidx:05d}.jpg', f'{CHUNK_DIR}/{i:05d}.jpg')

    with torch.inference_mode(), autocast_ctx(device):
        state = predictor.init_state(video_path=CHUNK_DIR, offload_video_to_cpu=True)
        predictor.reset_state(state)

        prompts_added = False

        if chunk_kfs:
            for fidx in chunk_kfs:
                kf  = kf_by_frame[fidx]
                rel = fidx - chunk_start
                for part, obj_id in PART_IDS.items():
                    pt = kf[part]
                    if pt is not None:
                        predictor.add_new_points_or_box(
                            inference_state=state, frame_idx=rel, obj_id=obj_id,
                            points=np.array([pt], dtype=np.float32),
                            labels=np.array([1], dtype=np.int32),
                        )
                    else:
                        predictor.add_new_points_or_box(
                            inference_state=state, frame_idx=rel, obj_id=obj_id,
                            points=np.array([[cx_mid, cy_mid]], dtype=np.float32),
                            labels=np.array([0], dtype=np.int32),
                        )
            prompts_added = True

        elif carry_masks is not None:
            for part, obj_id in PART_IDS.items():
                mask = carry_masks.get(part)
                if mask is not None and mask.any():
                    predictor.add_new_mask(
                        inference_state=state, frame_idx=0, obj_id=obj_id,
                        mask=torch.from_numpy(mask),
                    )
                    prompts_added = True

        if not prompts_added:
            print(f'  → no bird, skipping.')
            predictor.reset_state(state)
            del state
            gc.collect()
            torch.cuda.empty_cache()
            shutil.rmtree(CHUNK_DIR)
            carry_masks = None
            continue

        for rel, obj_ids, logits in predictor.propagate_in_video(
                state, start_frame_idx=anchor_rel):
            orig = chunk_start + rel
            video_segments[orig] = {
                ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
                for i, oid in enumerate(obj_ids)
            }
            del logits

        if anchor_rel > 0:
            for rel, obj_ids, logits in predictor.propagate_in_video(
                    state, start_frame_idx=anchor_rel, reverse=True):
                orig = chunk_start + rel
                video_segments[orig] = {
                    ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
                    for i, oid in enumerate(obj_ids)
                }
                del logits

        last_orig = chunk[-1]
        if last_orig in video_segments:
            last_masks = video_segments[last_orig]
            carry_masks = (
                {p: last_masks[p].copy() for p in last_masks}
                if any(m.any() for m in last_masks.values())
                else None
            )

        predictor.reset_state(state)
        del state

    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(CHUNK_DIR)
    print(f'  → {sum(1 for f in chunk if f in video_segments)}/{len(chunk)} frames done'
          f'  |  total: {len(video_segments)}')

print(f'\nDone — {len(video_segments)} / {len(all_frames)} frames segmented.')

In [ ]:
# ── Cell 10: Save masks to Google Drive ──────────────────────────────────────
mask_dict = {}
for fidx, parts in video_segments.items():
    for part, mask in parts.items():
        mask_dict[f'{fidx:05d}_{part}'] = mask

mask_dict['_ann_frame'] = np.array([KEYFRAMES[0]['frame']])
mask_dict['_fps']       = np.array([fps])
mask_dict['_frame_wh']  = np.array([width, height])

np.savez_compressed(MASKS_OUT, **mask_dict)

size_mb = os.path.getsize(MASKS_OUT) / 1e6
print(f'Saved {len(video_segments)} frames × {len(PART_IDS)} masks → {MASKS_OUT}')
print(f'File size: {size_mb:.1f} MB')
print()
print('Next steps:')
print('  1. Go to drive.google.com')
print('  2. Download hummingbird_masks.npz to your Mac project folder')
print('  3. Run hummingbird_analysis_local.ipynb locally — no Colab needed again')